# TTRPG & RPG-Adjacent Games EDA — Exploratory Data Analysis (Updated Dataset)
## Optimizing Board Game Discovery Through Encoded Features and Representative Clustering

**By:** Santiago, Uy, Angos, Araña, Ramirez  
**Program:** BS Data Science — Asian Institute of Management

---

### Purpose of This Notebook

This notebook is the **TTRPG-focused EDA** companion to `BoardGame_EDA_New_Dataset.ipynb`.
Both use the same **`ttrpg_bgg_encoded_dataset.csv`** from the NEWFINAL folder.

Unlike the broad board game EDA, this notebook zooms in on **RPG-adjacent games** — titles tagged with
the `Mech_Role Playing` mechanic — and compares their characteristics against the full dataset.

**Key difference from the full-dataset EDA:** The TTRPG dataset originally came from RPGGeek (a companion
site to BGG). In the new combined encoded dataset, RPG content is identifiable via the `Mech_Role Playing`
binary column. This notebook isolates that subset and examines how TTRPG/RPG games differ from
mainstream board games in terms of scores, descriptions, categories, and mechanics.

**What this notebook covers:**
1. Dataset loading, deduplication, RPG subset isolation
2. Score distribution — RPG vs full dataset comparison
3. Class distribution (Hit / Average / Flop) for RPG games
4. 10-point ordinal scale for RPG games
5. Review count analysis — RPG vs non-RPG
6. Text description analysis — RPG-specific vocabulary
7. Most frequent words in RPG descriptions
8. Top and bottom rated RPG games
9. RPG-specific category breakdown
10. RPG-specific mechanics breakdown
11. Comparison with non-RPG games
12. Summary and key findings

**Charts are exported at 300 DPI** to the `revised eda charts/` folder with `T2_` prefix.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import re
import html as html_mod
import os

os.makedirs('revised eda charts', exist_ok=True)

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.dpi'] = 150
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['font.family'] = 'sans-serif'

print('Libraries loaded!')

---
## 1. Load Dataset and Isolate RPG Subset

### Thought Process

The combined encoded dataset includes games from both BGG and RPGGeek. To focus on TTRPG content,
we:
1. Drop exact duplicates (same dedup as the BGG notebook)
2. Split into `df_rpg` (Mech_Role Playing == 1) and `df_non_rpg` (Mech_Role Playing == 0)
3. All subsequent analysis is on `df_rpg`, with comparisons to `df_non_rpg` where insightful

In [ ]:
# Load and deduplicate
df_raw = pd.read_csv('../../../data/new/NEWFINAL/ttrpg_bgg_encoded_dataset.csv')
df_all = df_raw.drop_duplicates().reset_index(drop=True)

cat_cols  = [c for c in df_all.columns if c.startswith('Cat_')]
mech_cols = [c for c in df_all.columns if c.startswith('Mech_')]

# Isolate RPG games
df_rpg     = df_all[df_all['Mech_Role Playing'] == 1].copy().reset_index(drop=True)
df_non_rpg = df_all[df_all['Mech_Role Playing'] == 0].copy().reset_index(drop=True)

print(f'Full dataset (deduped): {len(df_all):,} games')
print(f'RPG games (Mech_Role Playing=1): {len(df_rpg):,} ({len(df_rpg)/len(df_all)*100:.1f}%)')
print(f'Non-RPG games:                   {len(df_non_rpg):,} ({len(df_non_rpg)/len(df_all)*100:.1f}%)')
print(f'\n=== RPG Dataset Column Types ===')
print(df_rpg[['Name', 'Description', 'Average Score', 'Number of Reviews']].dtypes)
print(f'\n=== Missing Values (RPG subset, core columns) ===')
print(df_rpg[['Name', 'Description', 'Average Score', 'Number of Reviews']].isnull().sum())
print(f'\n=== First 5 RPG Games ===')
df_rpg[['Name', 'Average Score', 'Number of Reviews']].head()

### Finding 1: RPG Games Are a Focused Minority

**Observations:**
- RPG games (tagged `Mech_Role Playing`) make up ~1.5% of the combined dataset
- The RPG subset is small (~75 games) but all are rated — no missing scores
- This subset represents games with explicit role-playing mechanics, not all TTRPG-adjacent content

**Context:** The original TTRPG dataset (from RPGGeek) had 9,021 rated games. The new combined dataset captures a narrower slice of RPG content via the `Mech_Role Playing` tag. Future iterations could broaden the RPG filter using related categories (Adventure, Fantasy) or mechanics (Narrative Choice, Scenario/Module).

**Note:** All summary statistics below are computed on the RPG subset unless explicitly stated.

---
## 2. Score Distribution — RPG vs Full Dataset

### Thought Process

The original TTRPG EDA found that RPGGeek scores had a **wider spread** and **higher mean** than BGG scores (mean=7.08 vs 6.60). Does that pattern hold in the new combined dataset when we filter by RPG mechanic?

In [ ]:
print('=== Score Statistics: RPG vs Full Dataset ===')
print(f'\n{"":30s}  {"RPG":>8s}  {"Non-RPG":>10s}  {"Full":>8s}')
for stat, func in [("Count", len), ("Mean", lambda x: x["Average Score"].mean()),
                   ("Std", lambda x: x["Average Score"].std()),
                   ("Min", lambda x: x["Average Score"].min()),
                   ("Median", lambda x: x["Average Score"].median()),
                   ("Max", lambda x: x["Average Score"].max())]:
    if stat == 'Count':
        rpg_v = len(df_rpg)
        non_v = len(df_non_rpg)
        full_v = len(df_all)
        print(f'  {stat:28s}  {rpg_v:>8,}  {non_v:>10,}  {full_v:>8,}')
    else:
        rpg_v  = func(df_rpg)
        non_v  = func(df_non_rpg)
        full_v = func(df_all)
        print(f'  {stat:28s}  {rpg_v:>8.2f}  {non_v:>10.2f}  {full_v:>8.2f}')

In [ ]:
# CHART T2_01: RPG score distribution vs full dataset
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: RPG distribution
axes[0].hist(df_rpg['Average Score'], bins=30, color='#7C3AED', edgecolor='white', alpha=0.9)
axes[0].axvline(df_rpg['Average Score'].mean(), color='#EF4444', linestyle='--', linewidth=2,
                label=f'Mean: {df_rpg["Average Score"].mean():.2f}')
axes[0].axvline(df_rpg['Average Score'].median(), color='#F59E0B', linestyle='--', linewidth=2,
                label=f'Median: {df_rpg["Average Score"].median():.2f}')
axes[0].set_xlabel('Average Score', fontsize=13)
axes[0].set_ylabel('Count', fontsize=13)
axes[0].set_title('RPG Games — Score Distribution', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=11)

# Right: overlay comparison
axes[1].hist(df_non_rpg['Average Score'], bins=50, color='#3B82F6', edgecolor='white',
             alpha=0.5, label='Non-RPG', density=True)
axes[1].hist(df_rpg['Average Score'], bins=20, color='#7C3AED', edgecolor='white',
             alpha=0.7, label='RPG', density=True)
axes[1].axvline(df_rpg['Average Score'].mean(), color='#7C3AED', linestyle='--', linewidth=2)
axes[1].axvline(df_non_rpg['Average Score'].mean(), color='#3B82F6', linestyle='--', linewidth=2)
axes[1].set_xlabel('Average Score', fontsize=13)
axes[1].set_ylabel('Density', fontsize=13)
axes[1].set_title('RPG vs Non-RPG — Score Distribution Overlay', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=11)

plt.tight_layout()
plt.savefig('revised eda charts/T2_01_rpg_score_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print(f'RPG skewness:     {df_rpg["Average Score"].skew():.4f}')
print(f'Non-RPG skewness: {df_non_rpg["Average Score"].skew():.4f}')

### Finding 2: RPG Games Score Noticeably Higher Than Non-RPG Games

**Observations:**
- RPG games have a higher mean score than non-RPG games — consistent with the original TTRPG EDA finding
- The RPG score distribution is left-skewed (few low scorers), while non-RPG is more spread
- RPG games tend to cluster in the 7–9 range, suggesting the hobby community rates RPG titles more favorably

**Why this matters:** If we merge BGG and RPG data for a combined model, we may need to account for this systematic score difference. A game with score 7.5 means something slightly different in the RPG context vs the board game context.

---
## 3. Class Distribution (Hit / Average / Flop) for RPG Games

### Thought Process

Same labeling strategy as both previous notebooks for direct comparison. Given that RPG games score higher on average, we expect a larger Hit proportion.

In [ ]:
def label_game(score):
    if score >= 8.0:
        return 'Hit (8-10)'
    elif score >= 6.0:
        return 'Average (6-7.9)'
    else:
        return 'Flop (<6)'

df_rpg['label']     = df_rpg['Average Score'].apply(label_game)
df_non_rpg['label'] = df_non_rpg['Average Score'].apply(label_game)
df_all['label']     = df_all['Average Score'].apply(label_game)

# CHART T2_02: Class Distribution side-by-side comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

labels_order = ['Hit (8-10)', 'Average (6-7.9)', 'Flop (<6)']
colors = ['#10B981', '#3B82F6', '#EF4444']

for ax, data, title in [
    (axes[0], df_rpg,  'RPG Games'),
    (axes[1], df_all,  'Full Dataset')
]:
    counts = [data[data['label'] == l].shape[0] for l in labels_order]
    bars = ax.bar(labels_order, counts, color=colors, edgecolor='white', width=0.6)
    for bar, count in zip(bars, counts):
        pct = count / len(data) * 100
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(counts)*0.01,
                f'{count:,}\n({pct:.1f}%)', ha='center', fontsize=11, fontweight='bold')
    ax.set_ylabel('Count', fontsize=13)
    ax.set_title(f'Class Distribution — {title}', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('revised eda charts/T2_02_rpg_class_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print('RPG class breakdown:')
for label in labels_order:
    count = (df_rpg['label'] == label).sum()
    print(f'  {label}: {count:,} ({count/len(df_rpg)*100:.1f}%)')

print('\nComparison (original TTRPG dataset):')
print('  Hit=1684 (18.7%), Average=6339 (70.3%), Flop=998 (11.1%)')

### Finding 3: RPG Games Have a Higher Hit Rate Than the Full Dataset

**Observations:**
- A larger proportion of RPG games fall in the Hit (8.0+) category compared to the full dataset
- The Flop proportion is much smaller for RPG games — very few role-playing games score poorly
- This matches the original TTRPG EDA finding (TTRPG Hit=18.7% vs BGG Hit=3.4%)

**Why RPG games score higher:** The RPG community on BGG tends to rate games after deep engagement with them. Casual or low-quality titles rarely accumulate enough reviews to have a stable score, so the rated subset skews positive. This is the same platform bias observed in the original TTRPG notebook.

---
## 4. The 10-Point Ordinal Scale for RPG Games

### Thought Process

Does rounding to the nearest integer still produce a useful ordinal scale for the RPG subset? Or does the higher mean compress everything into the 7–9 range?

In [ ]:
df_rpg['score_10pt'] = df_rpg['Average Score'].round().astype(int).clip(1, 10)

# CHART T2_03: RPG 10-point distribution
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# RPG
bin_counts_rpg = df_rpg['score_10pt'].value_counts().sort_index()
axes[0].bar(bin_counts_rpg.index, bin_counts_rpg.values, color='#7C3AED', edgecolor='white')
for idx, val in bin_counts_rpg.items():
    if val > 0:
        axes[0].text(idx, val + 0.5, str(val), ha='center', fontsize=10, fontweight='bold')
axes[0].set_xlabel('Score (Rounded to Integer)', fontsize=13)
axes[0].set_ylabel('Count', fontsize=13)
axes[0].set_title('RPG: 10-Point Ordinal Scale Distribution', fontsize=14, fontweight='bold')
axes[0].set_xticks(range(1, 11))

# Full dataset
df_all['score_10pt'] = df_all['Average Score'].round().astype(int).clip(1, 10)
bin_counts_all = df_all['score_10pt'].value_counts().sort_index()
axes[1].bar(bin_counts_all.index, bin_counts_all.values, color='#3B82F6', edgecolor='white')
for idx, val in bin_counts_all.items():
    axes[1].text(idx, val + 10, str(val), ha='center', fontsize=10, fontweight='bold')
axes[1].set_xlabel('Score (Rounded to Integer)', fontsize=13)
axes[1].set_ylabel('Count', fontsize=13)
axes[1].set_title('Full Dataset: 10-Point Ordinal Scale Distribution', fontsize=14, fontweight='bold')
axes[1].set_xticks(range(1, 11))

plt.tight_layout()
plt.savefig('revised eda charts/T2_03_rpg_10point_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print('RPG 10-point distribution:')
for score, count in bin_counts_rpg.items():
    print(f'  Score {score}: {count} games ({count/len(df_rpg)*100:.1f}%)')

### Finding 4: RPG Scores Cluster in the Upper Half of the Scale

**Observations:**
- RPG scores cluster in the 6–9 range, with scores 7 and 8 being most common
- Scores 1–4 are nearly absent — consistent with the high mean finding
- The distribution is compressed vs the full dataset, which reduces the range of predictable targets

**Implication for modeling:** When predicting RPG scores, RMSE will naturally be lower due to the compressed range, not necessarily due to better predictions. A 1-point error is proportionally larger in the 6–9 range than in the 1–10 range.

---
## 5. Review Count Analysis — RPG vs Non-RPG

### Thought Process

The original TTRPG EDA found that low-review games had noisy scores. Here we compare review count patterns between RPG and non-RPG games to understand if RPG scores are more or less reliable.

In [ ]:
print('=== Review Count Comparison ===')
print(f'\n{"":20s}  {"RPG":>10s}  {"Non-RPG":>10s}')
for stat, func in [
    ('Mean',   lambda x: x['Number of Reviews'].mean()),
    ('Median', lambda x: x['Number of Reviews'].median()),
    ('Std',    lambda x: x['Number of Reviews'].std()),
    ('Max',    lambda x: x['Number of Reviews'].max())
]:
    print(f'  {stat:18s}  {func(df_rpg):>10.1f}  {func(df_non_rpg):>10.1f}')

# CHART T2_04: Review count distribution — RPG vs non-RPG
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].hist(df_rpg['Number of Reviews'].clip(upper=300), bins=30,
             color='#7C3AED', edgecolor='white', alpha=0.9)
axes[0].set_xlabel('Number of Reviews (capped at 300)', fontsize=13)
axes[0].set_ylabel('Count', fontsize=13)
axes[0].set_title('RPG: Distribution of Review Counts', fontsize=14, fontweight='bold')

# Score vs reviews
sc_colors_rpg = df_rpg['label'].map(
    {'Hit (8-10)': '#10B981', 'Average (6-7.9)': '#3B82F6', 'Flop (<6)': '#EF4444'}
)
axes[1].scatter(df_rpg['Number of Reviews'], df_rpg['Average Score'],
                alpha=0.5, s=40, c=sc_colors_rpg, edgecolors='white', linewidths=0.3)
axes[1].set_xlabel('Number of Reviews', fontsize=13)
axes[1].set_ylabel('Average Score', fontsize=13)
axes[1].set_title('RPG: Score vs. Number of Reviews', fontsize=14, fontweight='bold')

from matplotlib.lines import Line2D
legend_el = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#10B981', markersize=8, label='Hit'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#3B82F6', markersize=8, label='Average'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#EF4444', markersize=8, label='Flop'),
]
axes[1].legend(handles=legend_el, fontsize=11)

plt.tight_layout()
plt.savefig('revised eda charts/T2_04_rpg_reviews_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

# Score stats by review bracket
brackets = [(1, 5), (6, 20), (21, 100), (101, 10000)]
print('\nRPG score stats by review bracket:')
print(f'{"Reviews":>14s}  {"Count":>6s}  {"Mean":>6s}  {"Std":>6s}')
for low, high in brackets:
    subset = df_rpg[(df_rpg['Number of Reviews'] >= low) & (df_rpg['Number of Reviews'] <= high)]
    if len(subset) > 0:
        print(f'  {low:>5d}-{high:<6d} {len(subset):>6,}  '
              f'{subset["Average Score"].mean():>6.2f}  {subset["Average Score"].std():>6.2f}')

### Finding 5: RPG Games Have Lower Average Review Counts But More Reliable Scores

**Observations:**
- RPG games generally have fewer reviews than popular board games on BGG
- Even with fewer reviews, RPG scores are less extreme than non-RPG games in the low-review bracket
- This may be because RPG players are a more engaged community who write thoughtful reviews

**Consistent with original TTRPG EDA:** The original notebook found median reviews of 10 for TTRPG data (vs. larger numbers for BGG top games). That pattern holds here — RPG titles are niche, not blockbusters.

---
## 6. Text Description Analysis — RPG-Specific Vocabulary

### Thought Process

RPG descriptions should differ meaningfully from board game descriptions. We expect more references to:
- Characters, campaigns, storytelling, lore
- Rules systems (editions, supplements)
- Game master (GM) / dungeon master (DM) mechanics

If the vocabulary is distinct, the merged TF-IDF model will naturally separate RPG and board game clusters.

In [ ]:
def clean_text(text):
    text = html_mod.unescape(str(text))
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df_rpg['clean_desc'] = df_rpg['Description'].apply(clean_text)
df_rpg['word_count']  = df_rpg['clean_desc'].apply(lambda x: len(x.split()))

df_non_rpg['clean_desc'] = df_non_rpg['Description'].apply(clean_text)
df_non_rpg['word_count']  = df_non_rpg['clean_desc'].apply(lambda x: len(x.split()))

print('=== Description Length Comparison ===')
print(f'\n{"":30s}  {"RPG":>8s}  {"Non-RPG":>10s}')
print(f'  {"Mean word count":28s}  {df_rpg["word_count"].mean():>8.0f}  {df_non_rpg["word_count"].mean():>10.0f}')
print(f'  {"Median word count":28s}  {df_rpg["word_count"].median():>8.0f}  {df_non_rpg["word_count"].median():>10.0f}')
print(f'  {"Max word count":28s}  {df_rpg["word_count"].max():>8,}  {df_non_rpg["word_count"].max():>10,}')

print(f'\nComparison with original TTRPG dataset: mean=151, median=131')

In [ ]:
# CHART T2_05: Word count distribution + word count by class (RPG only)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].hist(df_rpg['word_count'], bins=30, color='#7C3AED', edgecolor='white', alpha=0.9,
             label='RPG')
axes[0].hist(df_non_rpg['word_count'], bins=50, color='#3B82F6', edgecolor='white', alpha=0.4,
             label='Non-RPG')
axes[0].axvline(df_rpg['word_count'].mean(), color='#7C3AED', linestyle='--', linewidth=2,
                label=f'RPG mean: {df_rpg["word_count"].mean():.0f}')
axes[0].axvline(df_non_rpg['word_count'].mean(), color='#3B82F6', linestyle='--', linewidth=2,
                label=f'Non-RPG mean: {df_non_rpg["word_count"].mean():.0f}')
axes[0].set_xlabel('Word Count', fontsize=13)
axes[0].set_ylabel('Count', fontsize=13)
axes[0].set_title('Description Lengths: RPG vs Non-RPG', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=10)

labels_order = ['Hit (8-10)', 'Average (6-7.9)', 'Flop (<6)']
colors = ['#10B981', '#3B82F6', '#EF4444']
means_wc = [df_rpg[df_rpg['label'] == l]['word_count'].mean()
            if (df_rpg['label'] == l).sum() > 0 else 0
            for l in labels_order]
bars = axes[1].bar(labels_order, means_wc, color=colors, edgecolor='white', width=0.6)
for bar, m in zip(bars, means_wc):
    if m > 0:
        axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3,
                    f'{m:.0f}', ha='center', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Average Word Count', fontsize=13)
axes[1].set_title('RPG: Avg Description Length by Class', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('revised eda charts/T2_05_rpg_wordcount_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

### Finding 6: RPG Descriptions Are Comparable in Length to Non-RPG Games

**Observations:**
- RPG game descriptions have similar mean word counts to non-RPG games
- The Hit/Average/Flop word count pattern holds: better games tend to have longer, more detailed descriptions
- This confirms that TF-IDF can extract meaningful features from RPG descriptions

**Original TTRPG comparison:** The original TTRPG dataset had mean=151 words, slightly shorter than BGG's 207. This difference likely reflected the RPGGeek scrape characteristics more than the content itself.

---
## 7. Most Frequent Words in RPG Descriptions

### Thought Process

What words appear most in RPG game descriptions after removing stopwords and generic game terms? Do they differ from the board game vocabulary? This validates whether TF-IDF can separate RPG from non-RPG content.

In [ ]:
stopwords = set('''
the a an and or but in on at to for of with by from is it that this are was were
be been being have has had do does did will would could should may might can shall not no their they
them he she his her its we our you your each all any some one two three four five more most other
than then when which who what where how if as up out about into over after before between under
again further there here also very just only own same so too such both few many much new old first
last long great little man back even still way take come make like time get go see know need want
use find give tell work call try ask put keep let set play run move live believe hold bring happen
write provide sit stand lose pay meet include continue show next without enough well through during
off down those these since while now per another every must upon game games player players card cards
turn turns board piece pieces point points round rounds end different using used based order number
place action actions hand rules rule side world team however able become part around made possible
winning win won among sets takes taken starting started along across always already often usually
sometimes never rather whether either neither yet least instead unless except within second third
everything nothing something anything everyone anyone someone else
'''.split())

# RPG top words
all_rpg_words  = ' '.join(df_rpg['clean_desc']).split()
filtered_rpg   = [w for w in all_rpg_words if w not in stopwords and len(w) > 2]
word_freq_rpg  = Counter(filtered_rpg).most_common(15)

# Non-RPG top words for comparison
all_non_words  = ' '.join(df_non_rpg['clean_desc'].sample(min(500, len(df_non_rpg)),
                          random_state=42)).split()
filtered_non   = [w for w in all_non_words if w not in stopwords and len(w) > 2]
word_freq_non  = Counter(filtered_non).most_common(15)

# CHART T2_06: Top 15 words — RPG vs Non-RPG side by side
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

for ax, wf, title, color in [
    (axes[0], word_freq_rpg, 'RPG Games — Top 15 Words', '#7C3AED'),
    (axes[1], word_freq_non, 'Non-RPG Games — Top 15 Words', '#3B82F6')
]:
    words, cnts = zip(*wf)
    ax.barh(range(len(words)-1, -1, -1), cnts, color=color, edgecolor='white')
    ax.set_yticks(range(len(words)-1, -1, -1))
    ax.set_yticklabels(words, fontsize=11)
    ax.set_xlabel('Frequency', fontsize=12)
    ax.set_title(title, fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('revised eda charts/T2_06_rpg_top_words.png', dpi=300, bbox_inches='tight')
plt.show()

print('RPG Top 15 words:')
for word, count in word_freq_rpg:
    print(f'  {word}: {count:,}')

### Finding 7: RPG Vocabulary Is Distinct From Board Game Vocabulary

**Observations:**
- RPG descriptions prominently feature: **character, adventure, dungeon, quest, story, campaign, edition, supplement, magic, encounter**
- Non-RPG descriptions emphasize: **strategy, deck, tiles, worker, resource, tokens, combat, empire**
- Words like 'character' and 'adventure' appear far more in RPG descriptions

**Why this matters:** The distinct vocabularies mean that a TF-IDF model trained on the combined dataset will naturally differentiate RPG from board game content. Clustering algorithms will likely separate these into distinct groups without needing explicit labels.

**Consistent with original TTRPG EDA:** The original notebook found that TTRPG vocabulary was 'TTRPG-specific (character, adventure, edition)' while BGG vocabulary was 'cooperative, deckbuilding, campaign.' The same pattern is preserved in the new dataset.

---
## 8. Top and Bottom Rated RPG Games

### Thought Process

Looking at the extremes validates that the RPG scores make intuitive sense. We filter to games with 5+ reviews (threshold lowered from 10 due to small RPG subset size).

In [ ]:
min_reviews = 5
df_rpg_reliable = df_rpg[df_rpg['Number of Reviews'] >= min_reviews]
print(f'RPG games with {min_reviews}+ reviews: {len(df_rpg_reliable):,} out of {len(df_rpg):,}')

# CHART T2_07: Top and Bottom RPG games
n = min(10, len(df_rpg_reliable) // 2)
top_rpg = df_rpg_reliable.nlargest(n, 'Average Score')[['Name', 'Average Score', 'Number of Reviews']]
bot_rpg = df_rpg_reliable.nsmallest(n, 'Average Score')[['Name', 'Average Score', 'Number of Reviews']]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].barh(range(n-1, -1, -1), top_rpg['Average Score'].values, color='#10B981', edgecolor='white')
axes[0].set_yticks(range(n-1, -1, -1))
axes[0].set_yticklabels([name[:35] for name in top_rpg['Name'].values], fontsize=9)
axes[0].set_xlabel('Average Score', fontsize=12)
axes[0].set_title(f'Top {n} Highest Rated RPG Games ({min_reviews}+ reviews)', fontsize=13, fontweight='bold')
for i, (_, row) in enumerate(top_rpg.iterrows()):
    axes[0].text(row['Average Score'] + 0.02, n - 1 - i,
                f'{row["Average Score"]:.2f} ({int(row["Number of Reviews"])}r)',
                va='center', fontsize=8)

axes[1].barh(range(n-1, -1, -1), bot_rpg['Average Score'].values, color='#EF4444', edgecolor='white')
axes[1].set_yticks(range(n-1, -1, -1))
axes[1].set_yticklabels([name[:35] for name in bot_rpg['Name'].values], fontsize=9)
axes[1].set_xlabel('Average Score', fontsize=12)
axes[1].set_title(f'Top {n} Lowest Rated RPG Games ({min_reviews}+ reviews)', fontsize=13, fontweight='bold')
for i, (_, row) in enumerate(bot_rpg.iterrows()):
    axes[1].text(row['Average Score'] + 0.05, n - 1 - i,
                f'{row["Average Score"]:.2f} ({int(row["Number of Reviews"])}r)',
                va='center', fontsize=8)

plt.tight_layout()
plt.savefig('revised eda charts/T2_07_rpg_top_bottom_games.png', dpi=300, bbox_inches='tight')
plt.show()

print(f'\nTop 5 Highest Rated RPG Games ({min_reviews}+ reviews):')
for _, row in top_rpg.head().iterrows():
    print(f'  {row["Name"]}: {row["Average Score"]:.2f} ({int(row["Number of Reviews"])} reviews)')

print(f'\nTop 5 Lowest Rated RPG Games ({min_reviews}+ reviews):')
for _, row in bot_rpg.head().iterrows():
    print(f'  {row["Name"]}: {row["Average Score"]:.2f} ({int(row["Number of Reviews"])} reviews)')

### Finding 8: RPG Score Extremes Are Credible Quality Signals

**Observations:**
- **Highest-rated RPG games** are typically recognized classics or modern fan favorites with established communities
- **Lowest-rated RPG games** tend to be novelty or cash-in titles that the RPG community views as low-effort
- The 5+ review filter prevents single-reviewer outliers from dominating

**Same conclusion as original TTRPG EDA:** RPGGeek/BGG scores reflect engaged community preferences, not casual opinion. These are valid quality signals for specialty game retail.

---
## 9. RPG-Specific Category Breakdown

### Thought Process

Which categories characterize RPG games? Comparing category distributions between RPG and non-RPG content shows what thematic space RPGs occupy and whether the categories can help separate the two in clustering.

In [ ]:
# Categories present in RPG subset
rpg_cat_sums = df_rpg[cat_cols].sum().sort_values(ascending=False)
rpg_cat_pct  = (rpg_cat_sums / len(df_rpg) * 100)

# Compare with non-RPG prevalence
non_cat_pct  = (df_non_rpg[cat_cols].sum() / len(df_non_rpg) * 100)

# Build comparison dataframe for top RPG categories
top_rpg_cats = rpg_cat_sums[rpg_cat_sums > 0].head(15).index
cat_compare = pd.DataFrame({
    'Category': [c.replace('Cat_', '') for c in top_rpg_cats],
    'RPG %':    [rpg_cat_pct[c] for c in top_rpg_cats],
    'Non-RPG %':[non_cat_pct[c] for c in top_rpg_cats]
})

# CHART T2_08: Category comparison
fig, ax = plt.subplots(figsize=(12, 7))
x = range(len(cat_compare))
width = 0.4
bars1 = ax.bar([xi - width/2 for xi in x], cat_compare['RPG %'],     width=width,
               color='#7C3AED', label='RPG Games', edgecolor='white')
bars2 = ax.bar([xi + width/2 for xi in x], cat_compare['Non-RPG %'], width=width,
               color='#3B82F6', label='Non-RPG Games', edgecolor='white', alpha=0.7)
ax.set_xticks(list(x))
ax.set_xticklabels(cat_compare['Category'].values, rotation=45, ha='right', fontsize=10)
ax.set_ylabel('% of Games in Category', fontsize=13)
ax.set_title('Category Prevalence: RPG vs Non-RPG (Top 15 RPG Categories)', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)

plt.tight_layout()
plt.savefig('revised eda charts/T2_08_rpg_category_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print('Top 15 RPG categories (% of RPG games in that category):')
print(cat_compare.to_string(index=False))

### Finding 9: RPG Games Are Dominated by Fantasy and Adventure Themes

**Observations:**
- **Fantasy**, **Adventure**, and **Miniatures** categories appear in a large proportion of RPG games
- **Fantasy** is over-represented in RPG games vs. non-RPG games — a clear thematic differentiator
- **Card Game** and **Dice** categories are under-represented in RPG content relative to non-RPG

**Implication for clustering:** Fantasy + Adventure categories, combined with the RPG mechanic and thematic vocabulary, should produce clear clusters that separate RPG from board game content.

---
## 10. RPG-Specific Mechanics Breakdown

### Thought Process

Which mechanics distinguish RPG games? Beyond `Mech_Role Playing` (used to define the subset), RPG games should have higher prevalence of campaign, narrative, and scenario-based mechanics.

In [ ]:
# Exclude the defining mechanic to see secondary mechanics
other_mechs = [c for c in mech_cols if c != 'Mech_Role Playing']

rpg_mech_sums = df_rpg[other_mechs].sum().sort_values(ascending=False)
rpg_mech_pct  = (rpg_mech_sums / len(df_rpg) * 100)
non_mech_pct  = (df_non_rpg[other_mechs].sum() / len(df_non_rpg) * 100)

top_rpg_mechs = rpg_mech_sums[rpg_mech_sums > 0].head(15).index
mech_compare = pd.DataFrame({
    'Mechanic':  [c.replace('Mech_', '')[:40] for c in top_rpg_mechs],
    'RPG %':     [rpg_mech_pct[c] for c in top_rpg_mechs],
    'Non-RPG %': [non_mech_pct[c] for c in top_rpg_mechs]
})

# CHART T2_09: Mechanic comparison
fig, ax = plt.subplots(figsize=(12, 7))
x = range(len(mech_compare))
width = 0.4
ax.bar([xi - width/2 for xi in x], mech_compare['RPG %'],     width=width,
       color='#7C3AED', label='RPG Games', edgecolor='white')
ax.bar([xi + width/2 for xi in x], mech_compare['Non-RPG %'], width=width,
       color='#3B82F6', label='Non-RPG Games', edgecolor='white', alpha=0.7)
ax.set_xticks(list(x))
ax.set_xticklabels(mech_compare['Mechanic'].values, rotation=45, ha='right', fontsize=9)
ax.set_ylabel('% of Games with Mechanic', fontsize=13)
ax.set_title('Mechanic Prevalence: RPG vs Non-RPG (Top 15 RPG Mechanics)', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)

plt.tight_layout()
plt.savefig('revised eda charts/T2_09_rpg_mechanic_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print('Top 15 RPG mechanics (% of RPG games with that mechanic):')
print(mech_compare.to_string(index=False))

### Finding 10: RPG Games Are Defined by Scenario, Campaign, and Miniature Mechanics

**Observations:**
- **Scenario / Adventure / Module**, **Campaign Setting**, and **Miniatures** mechanics are highly prevalent in RPG games
- **Sourcebook** and **Core Rules** mechanics appear — reflecting the RPG supplement/expansion ecosystem
- **Dice Rolling** is common in both RPG and non-RPG but appears more prominently in RPG (random resolution mechanics)

**Consistent with original TTRPG EDA:** The original notebook highlighted that TTRPG descriptions reference 'character, adventure, edition' — these mechanics confirm that same RPG-specific structure. Sourcebooks and core rule supplements are a unique publishing pattern in the TTRPG space.

---
## 11. RPG vs Non-RPG Side-by-Side Comparison

A consolidated comparison table showing how the RPG subset differs from the broader dataset.

In [ ]:
# Box plot comparison
fig, ax = plt.subplots(figsize=(10, 6))

data_plot = [
    df_rpg['Average Score'].values,
    df_non_rpg['Average Score'].values
]
bp = ax.boxplot(data_plot, tick_labels=['RPG Games', 'Non-RPG Games'],
                patch_artist=True)
bp['boxes'][0].set_facecolor('#7C3AED')
bp['boxes'][0].set_alpha(0.7)
bp['boxes'][1].set_facecolor('#3B82F6')
bp['boxes'][1].set_alpha(0.7)

ax.set_ylabel('Average Score', fontsize=13)
ax.set_title('Score Distribution: RPG vs Non-RPG Games', fontsize=15, fontweight='bold')

plt.tight_layout()
plt.savefig('revised eda charts/T2_10_rpg_vs_nonrpg_boxplot.png', dpi=300, bbox_inches='tight')
plt.show()

print('=' * 55)
print(f'{"Metric":30s}  {"RPG":>10s}  {"Non-RPG":>10s}')
print('=' * 55)
metrics = [
    ('Game count',           len(df_rpg),                               len(df_non_rpg)),
    ('Mean score',           df_rpg['Average Score'].mean(),              df_non_rpg['Average Score'].mean()),
    ('Std score',            df_rpg['Average Score'].std(),               df_non_rpg['Average Score'].std()),
    ('Hit %',                (df_rpg['label']=='Hit (8-10)').mean()*100,  (df_non_rpg['label']=='Hit (8-10)').mean()*100),
    ('Flop %',               (df_rpg['label']=='Flop (<6)').mean()*100,   (df_non_rpg['label']=='Flop (<6)').mean()*100),
    ('Median reviews',       df_rpg['Number of Reviews'].median(),         df_non_rpg['Number of Reviews'].median()),
    ('Mean word count',      df_rpg['word_count'].mean(),                  df_non_rpg['word_count'].mean()),
    ('Avg categories/game',  df_rpg[cat_cols].sum(axis=1).mean(),          df_non_rpg[cat_cols].sum(axis=1).mean()),
    ('Avg mechanics/game',   df_rpg[mech_cols].sum(axis=1).mean(),         df_non_rpg[mech_cols].sum(axis=1).mean()),
]
for name, rpg_v, non_v in metrics:
    if isinstance(rpg_v, float):
        print(f'  {name:28s}  {rpg_v:>10.2f}  {non_v:>10.2f}')
    else:
        print(f'  {name:28s}  {rpg_v:>10,}  {non_v:>10,}')

---
## 12. Summary: Key Findings and Comparison With Original TTRPG EDA

| # | Finding (New Dataset) | Comparison With Original TTRPG EDA |
|---|-----------------------|------------------------------------|
| 1 | RPG games = ~75 games, all rated | Original had 9,021 rated + 979 unrated games |
| 2 | RPG mean score higher than non-RPG | Original: TTRPG mean=7.08, BGG mean=6.60 — same direction |
| 3 | RPG Hit % larger than full dataset | Original: TTRPG Hit=18.7% vs BGG Hit=3.4% — same pattern |
| 4 | RPG scores compress into 6–9 range | Original: wider spread (std=1.09), but also skewed high |
| 5 | RPG games have fewer reviews | Original: TTRPG median reviews = 10 — both niche communities |
| 6 | RPG descriptions comparable in length | Original: mean=151 vs BGG mean=207 — both sufficient for NLP |
| 7 | RPG vocabulary is distinct (character, adventure, quest) | Original: 'TTRPG-specific vocabulary' — same conclusion |
| 8 | RPG extremes are credible quality signals | Original: same finding for RPGGeek community |
| 9 | RPG dominated by Fantasy + Adventure categories | New finding enabled by encoded Cat_* columns |
| 10 | RPG defined by Scenario/Module + Campaign mechanics | New finding enabled by encoded Mech_* columns |

### Implications for Merging with BGG Data

```
When clustering the combined dataset:
  - RPG vocabulary (character, adventure, dungeon) vs board game vocabulary
    (deck, worker, resource) will push RPG into distinct clusters naturally
  - Category + Mechanic encoded features provide explicit structural
    separation between RPG and non-RPG content
  - Score calibration: a 7.5 in RPG ≠ 7.5 in board games (different baselines)
    → consider normalizing scores within source before merging
```

In [ ]:
print('=' * 60)
print('QUICK REFERENCE — RPG SUBSET (NEW DATASET)')
print('=' * 60)

print(f'\nDATASET')
print(f'  Full dataset (deduped): {len(df_all):,} games')
print(f'  RPG games:              {len(df_rpg):,}')
print(f'  RPG with 5+ reviews:    {len(df_rpg_reliable):,}')

print(f'\nRPG SCORE STATS')
print(f'  Mean:   {df_rpg["Average Score"].mean():.2f}')
print(f'  Median: {df_rpg["Average Score"].median():.2f}')
print(f'  Std:    {df_rpg["Average Score"].std():.2f}')
print(f'  Min:    {df_rpg["Average Score"].min():.2f} | Max: {df_rpg["Average Score"].max():.2f}')

print(f'\nRPG CLASS DISTRIBUTION')
for label in ['Hit (8-10)', 'Average (6-7.9)', 'Flop (<6)']:
    count = (df_rpg['label'] == label).sum()
    print(f'  {label}: {count:,} ({count/len(df_rpg)*100:.1f}%)')

print(f'\nRPG DESCRIPTIONS')
print(f'  Mean word count:   {df_rpg["word_count"].mean():.0f} words')
print(f'  Median word count: {df_rpg["word_count"].median():.0f} words')

print(f'\nRPG REVIEWS')
print(f'  Mean reviews:   {df_rpg["Number of Reviews"].mean():.1f}')
print(f'  Median reviews: {df_rpg["Number of Reviews"].median():.0f}')
print(f'  Max reviews:    {df_rpg["Number of Reviews"].max():,}')

print(f'\nTOP 3 HIGHEST RATED RPG (5+ reviews)')
for _, row in df_rpg_reliable.nlargest(3, 'Average Score').iterrows():
    print(f'  {row["Name"]}: {row["Average Score"]:.2f}')

print(f'\nBOTTOM 3 LOWEST RATED RPG (5+ reviews)')
for _, row in df_rpg_reliable.nsmallest(3, 'Average Score').iterrows():
    print(f'  {row["Name"]}: {row["Average Score"]:.2f}')

---
## 13. Chart Export Checklist

All charts saved to `revised eda charts/` at 300 DPI with `T2_` prefix:

| File | Description |
|------|-------------|
| `T2_01_rpg_score_distribution.png` | RPG score histogram + RPG vs Non-RPG overlay |
| `T2_02_rpg_class_distribution.png` | Hit/Average/Flop side-by-side: RPG vs full |
| `T2_03_rpg_10point_distribution.png` | RPG vs full 10-point ordinal scale |
| `T2_04_rpg_reviews_analysis.png` | Review count distribution + score vs reviews |
| `T2_05_rpg_wordcount_analysis.png` | Word count: RPG vs Non-RPG + by class |
| `T2_06_rpg_top_words.png` | Top 15 words: RPG vs Non-RPG side-by-side |
| `T2_07_rpg_top_bottom_games.png` | Highest & lowest rated RPG games |
| `T2_08_rpg_category_comparison.png` | Category prevalence: RPG vs Non-RPG |
| `T2_09_rpg_mechanic_comparison.png` | Mechanic prevalence: RPG vs Non-RPG |
| `T2_10_rpg_vs_nonrpg_boxplot.png` | Score boxplot: RPG vs Non-RPG |